# overlays

> Popovers, menus, dialogs, drawers, and tooltips

In [ ]:
#| default_exp overlays

An overlay sits above the page without changing its layout. Popovers provide light dismiss; dialogs make the page behind them inert. We keep those native behaviours, add a shared floating surface, and position popovers beside the control that opened them. Menus and tooltips build on that panel.

In [ ]:
from fasthtml.common import Div, Span, Button, A, P, Small, H2, Hr, FT, unqid
from answerui.registry import reg, hypelt
from answerui.layout import Stack, Cluster
from answerui.demo import example
import answerui.base  # the reset and tokens this styling is written on
import answerui.controls  # the buttons and menu items the panels hold


In [ ]:
from fasthtml.common import *
from answerui.demo import demo
from fastcore.test import *


## Popover

`Popover` is a floating neutral surface. A button opens it with `popovertarget`, and the browser supplies light dismiss and Escape. While the panel is open, every button that names it carries `aria-expanded`, so the one that opened it shows as pressed. Assistive technology already has that state from `popovertarget`; the attribute serves the stylesheet, and the `expanded` rule writes it when the panel toggles. The `anchored` marker keeps its styling separate from other uses of the native `popover` attribute.

CSS anchor positioning places the panel. A popover opened by `popovertarget` or `commandfor` has that button as its implicit anchor, so `position-area` puts the panel below the button, aligned to its start edge, with no anchor named on either element. The fallbacks flip it above, or to the other side, when the first position would overflow the viewport, and `position-try-order` gives a panel taller than either side the roomier one, where it scrolls, less a gap and an edge margin. An ancestor marked `anchor`, such as a navbar, is a named anchor: a panel inside it takes its vertical position from the bar's bottom edge instead, so a menu opens below the bar rather than across its padding. The browser follows scrolling, resizing, and changes in content size itself. An engine without anchor positioning gets the same placement from the popover's `placement` script, which runs only there.

In [ ]:
_fallback = "if (CSS.supports('position-area', 'center')) return;\n"
_popover_on = {
    ('button[popovertarget], button[commandfor]', 'click'): _fallback + r"""
const id = el.getAttribute('popovertarget') || el.getAttribute('commandfor');
const panel = document.getElementById(id);
if (panel?.matches('[popover][anchored]')) panel._anchor = el;""",
    ('&', 'beforetoggle', True): _fallback + r"""
if (e.target !== el) return;
if (e.newState === 'open') {
    if (e.source) el._anchor = e.source;
    el.style.visibility = 'hidden';
} else el._unwatch?.();""",
    ('&', 'toggle', True): _fallback + r"""
if (e.target !== el) return;
el._unwatch?.();
if (!el.matches(':popover-open')) return;
const anchor = el._anchor;
const place = () => {
    if (!anchor?.isConnected) { el.hidePopover(); return; }
    const r = anchor.getBoundingClientRect(), b = (anchor.closest('[anchor]') ?? anchor).getBoundingClientRect(), gap = 4, edge = 8;
    const vw = document.documentElement.getBoundingClientRect().width, vh = document.documentElement.clientHeight;
    const below = vh - b.bottom - gap - edge, above = b.top - gap - edge;
    const height = el.scrollHeight + el.offsetHeight - el.clientHeight;
    const up = height > below && above > below;
    el.style.setProperty('--_available-height', Math.max(0, up ? above : below) + 'px');
    const x = getComputedStyle(anchor).direction === 'rtl' ? r.right - el.offsetWidth : r.left;
    const y = up ? b.top - gap - el.offsetHeight : b.bottom + gap;
    el.style.left = Math.max(edge, Math.min(x, vw - el.offsetWidth - edge)) + 'px';
    el.style.top = Math.max(edge, Math.min(y, vh - el.offsetHeight - edge)) + 'px';
};
if (anchor) {
    place();
    const watch = new AbortController(), size = new ResizeObserver(place);
    addEventListener('resize', place, {signal: watch.signal});
    document.addEventListener('scroll', place, {capture: true, signal: watch.signal});
    size.observe(anchor); size.observe(el);
    el._unwatch = () => { watch.abort(); size.disconnect(); };
}
el.style.visibility = '';""",
    ('&', 'cleanup'): _fallback + "if (e.target === el) el._unwatch?.()"}

_popover_css = r"""
[popover][anchored] {
  --face: var(--n-2); background: var(--face); color: var(--n2);
  border: 1px solid var(--n-1); border-radius: var(--radius); padding: var(--space); box-shadow: var(--floating);
  position: fixed; inset: auto; margin: 0; max-inline-size: calc(100vw - 2 * var(--space)); overflow: auto;
  position-area: block-end span-inline-end; position-try-fallbacks: flip-block, flip-inline, flip-block flip-inline; position-try-order: most-block-size;
  margin-block-start: 4px; max-block-size: calc(100% - 12px);
}
[anchor] { anchor-name: --anchor; }
[anchor] [popover][anchored] { inset-block-start: anchor(--anchor bottom); }
@supports not (position-area: center) {
  [popover][anchored] { margin: 0; max-block-size: var(--_available-height, calc(100dvh - 2 * var(--space))); }
}
"""

@hypelt(sel='[popover][anchored]', layer='components', css=_popover_css)
def Popover(*c, popover='auto', **attrs):
    "A floating panel positioned beside the button that opens it"
    return Div(*c, popover=popover, anchored=True, **attrs)

Popover.rule('expanded', on={('[popover]', 'toggle', True): r"""
if (e.target !== el) return;
for (const b of document.querySelectorAll(`[popovertarget="${CSS.escape(el.id)}"]:not([popovertargetaction=hide]), [commandfor="${CSS.escape(el.id)}"]:is([command=toggle-popover], [command=show-popover])`)) b.setAttribute('aria-expanded', String(el.matches(':popover-open')));"""},
    layer='components', doc='Every button that opens or toggles the panel carries `aria-expanded` while it is open.')
Popover.rule('placement', on=_popover_on, layer='components', doc='In an engine without anchor positioning: remembers the button that opened the panel, places the panel beside it within the viewport, and follows scrolling and resizing until it closes.')

Popover.placement [components] [popover][anchored]: In an engine without anchor positioning: remembers the button that opened the panel, places the panel beside it within the viewport, and follows scrolling and resizing until it closes.

The panel fades in as it opens and out as it closes. Transitions on `display` and `overlay` keep it rendered, in the top layer, until the fade ends. A dialog and its backdrop fade the same way, so one rule serves the three:


In [ ]:
reg.rule('fade', r'''
:is(dialog, [popover][anchored]), dialog::backdrop {
  opacity: 0;
  transition: opacity var(--motion) var(--ease), translate var(--motion) var(--ease), display var(--motion) allow-discrete, overlay var(--motion) allow-discrete;
}
:is(dialog[open], [popover][anchored]:popover-open), dialog[open]::backdrop { opacity: 1; @starting-style { opacity: 0; } }
''', layer='components', doc='Dialogs, their backdrops, and anchored popovers fade over `--motion`, staying in the top layer until the fade ends.')

fade [components]: Dialogs, their backdrops, and anchored popovers fade over `--motion`, staying in the top layer until the fade ends.

Any button can open a popover by naming its id. The panel holds ordinary content:

In [ ]:
@example(Popover)
def _popover_example():
    id = unqid()
    return Cluster(Button('What is a workspace?', type='button', secondary=True, popovertarget=id),
        Popover(P("A workspace holds a team's projects, members, and billing."), id=id, style='max-inline-size: 18rem'))

In [ ]:
Popover.demo(height='9rem')

```html
<div cluster>
<button type="button" secondary popovertarget="_ct9UZLv9S6W7kozujWSbLQ">What is a workspace?</button>  <div popover="auto" anchored id="_ct9UZLv9S6W7kozujWSbLQ" style="max-inline-size: 18rem">
    <p>A workspace holds a team's projects, members, and billing.</p>
  </div>
</div>

```

## Dropdown

`Dropdown` returns a trigger and its popover of links, buttons, and separators. Other content, such as a heading naming the signed-in user, sits in line with the items. Give it text for a secondary trigger, or a configured `Button` to choose the appearance, accessible name, and other attributes. A trigger that shows only an avatar or an icon needs a name of its own, so pass a `Button` with an `aria-label`. The trigger is always `type='button'`, so opening it does not submit a surrounding form.

These are ordinary HTML controls, not an ARIA application menu: Tab and Shift+Tab move between the items, and a disabled item is skipped because it cannot take focus. Opening focuses the first enabled item, because `Dropdown` marks it `autofocus` and a popover focuses that element as it shows. Choosing a button closes the panel, because `Dropdown` gives each button `popovertarget` and `popovertargetaction="hide"` for it. Following a link closes it too, whether the browser or htmx handles the link, which is the `choose` rule. Escape and light dismiss remain native. An item marked `aria-disabled` does nothing, which is the `disabled` rule.

In [ ]:
_dropdown_disabled = {('& > [aria-disabled=true]', 'click', True): "e.preventDefault(); e.stopImmediatePropagation()"}

_dropdown_css = r"""
[popover][dropdown] { min-inline-size: min(12rem, calc(100vw - 2 * var(--space))); padding: var(--space-3); }
[popover][dropdown]:popover-open { display: grid; align-content: start; block-size: max-content; }
[dropdown] > :is(a, button) {
  display: block; inline-size: 100%; text-align: start; padding: var(--space-2) var(--space-1);
  font: inherit; font-weight: var(--weight); color: inherit; text-decoration: none;
  background: none; border: 0; border-radius: var(--radius-1); box-shadow: none;
}
[dropdown] > :is(a, button):not(:disabled, [aria-disabled=true]):is(:hover, :focus-visible) {
  background: var(--v-3); color: var(--v2);
}
[dropdown] > :is(a, button)[danger]:not(:disabled, [aria-disabled=true]) { color: var(--v); }
[dropdown] > :is(a, button):is(:disabled, [aria-disabled=true]) { color: var(--n1); cursor: default; }
[dropdown] > hr { margin-block: var(--space-3); }
[dropdown] > :not(a, button, hr) { padding-inline: var(--space-1); }
"""

@hypelt(sel='[popover][dropdown]', layer='components', css=_dropdown_css)
def Dropdown(
    trigger,  # Label text or a configured Button
    *items,   # Buttons, links, Hr separators, and other content such as a heading, which aligns with the items
    id=None,  # Panel id; generated when omitted
    **attrs,  # Attributes for the panel
):
    "A button and its popover of ordinary links and actions"
    id = id or unqid()
    if not isinstance(trigger, FT) or trigger.tag != 'button': trigger = Button(trigger, secondary=True)
    acts = [c for c in items if isinstance(c, FT) and c.tag in ('a', 'button')]
    for c in acts:
        if c.tag == 'button' and 'popovertarget' not in c.attrs: c(popovertarget=id, popovertargetaction='hide')
    live = [c for c in acts if (c.get('href') if c.tag == 'a' else not c.get('disabled')) and c.get('aria-disabled') != 'true' and not c.get('hidden')]
    if live: live[0](autofocus=True)
    return trigger(type='button', popovertarget=id), Popover(*items, dropdown=True, id=id, **attrs)

Dropdown.rule('disabled', on=_dropdown_disabled, layer='components', doc='An item marked aria-disabled does nothing.')
Dropdown.rule('choose', on={('& > a[href]', 'click'): "el.closest('[popover]').hidePopover()"}, layer='components',
    doc='Following a link closes the dropdown; a button closes it natively through popovertargetaction.')

Dropdown.choose [components] [popover][dropdown]: Following a link closes the dropdown; a button closes it natively through popovertargetaction.

The first enabled item carries `autofocus`, and every button names the panel so that pressing it hides the panel:

In [ ]:
trigger, panel = Dropdown('Account', Button('Duplicate', type='button', disabled=True), A('Profile', href='#'), Hr(), Button('Sign out', type='button'))
test_eq([x.get('autofocus') for x in panel.children], [None, True, None, None])
test_eq([x.get('popovertargetaction') for x in panel.children], ['hide', None, None, 'hide'])

The account menu opens with the user's name and email, in line with its items, and uses the default trigger; Options supplies its own button. Tab through the items: the heading is skipped, and so is the disabled Duplicate action.

In [ ]:
@example(Dropdown)
def _dropdown_example():
    trigger, panel = Dropdown(Button('Options', aria_label='Project actions'),
        Button('Rename', type='button'), Button('Duplicate', type='button', disabled=True),
        Hr(), Button('Delete', type='button', danger=True))
    assert trigger.get('aria-label') == 'Project actions'
    return Cluster(Dropdown('Account', P('Ada Lovelace'), Small('ada@example.com'), Hr(), A('Profile', href='#'), A('Billing', href='#'),
        Hr(), Button('Sign out', type='button')), trigger, panel)

In [ ]:
Dropdown.demo(height='14rem')

```html
<div cluster>
<button secondary type="button" popovertarget="_UeTegLlgReKLp63ESXBVYw">Account</button>  <div popover="auto" anchored dropdown id="_UeTegLlgReKLp63ESXBVYw">
    <p>Ada Lovelace</p>
<small>ada@example.com</small><hr><a href="#" autofocus>Profile</a><a href="#">Billing</a><hr><button type="button" popovertarget="_UeTegLlgReKLp63ESXBVYw" popovertargetaction="hide">Sign out</button>  </div>
<button aria-label="Project actions" type="button" popovertarget="_j1HK3OEyTkGk4FxIr4eTkw">Options</button>  <div popover="auto" anchored dropdown id="_j1HK3OEyTkGk4FxIr4eTkw">
<button type="button" popovertarget="_j1HK3OEyTkGk4FxIr4eTkw" popovertargetaction="hide" autofocus>Rename</button><button type="button" disabled popovertarget="_j1HK3OEyTkGk4FxIr4eTkw" popovertargetaction="hide">Duplicate</button><hr><button type="button" danger popovertarget="_j1HK3OEyTkGk4FxIr4eTkw" popovertargetaction="hide">Delete</button>  </div>
</div>

```

## Invoker commands

`commandfor` names a target and `command` says what to do: `show-modal`, `close`, or `toggle-popover`. Where the browser lacks invoker commands, one delegated rule calls the equivalent method. There is no constructor to learn.

In [ ]:
reg.rule('commands', sel='button[commandfor]', on={('&', 'click'): r"""
if ('commandForElement' in el) return;
const t = document.getElementById(el.getAttribute('commandfor')), c = el.getAttribute('command');
if (!t) return;
if (!['show-modal', 'close', 'toggle-popover'].includes(c)) return;
e.preventDefault();
if (c === 'show-modal') t.showModal();
else if (c === 'close') t.close();
else { t._anchor = el; t.togglePopover(); }"""},
    layer='components', doc='`command` and `commandfor` on engines without invoker commands.')

commands [components] button[commandfor]: `command` and `commandfor` on engines without invoker commands.

## Dialog

A modal is the platform's `dialog`, opened with `show-modal`. Focus moves into it, Escape closes it, and the page behind is inert, with nothing for us to script. The panel is the floating surface at the larger radius, and the backdrop is the ink at low alpha. Both fade over `--motion`. A dialog inherits the page's font and colours through its DOM parent, so it restates nothing. Closing is a button with `command="close"`, or a form with `method=\"dialog\"`. A click on the backdrop closes nothing unless the dialog carries `closedby=\"any\"`, which suits a drawer more than a confirmation:

In [ ]:
Dialog = hypelt('Dialog', layer='components', doc='A modal panel over an inert page; `drawer` slides it in from an edge.', css=r"""
dialog {
  --face: var(--n-2); background: var(--face); color: var(--n2);
  border: 1px solid var(--n-1); border-radius: var(--radius1); padding: var(--space1); box-shadow: var(--floating);
  inline-size: min(32rem, calc(100% - 2 * var(--space)));
  translate: 0 var(--space-2);
}
dialog[open] { translate: none; @starting-style { translate: 0 var(--space-2); } }
dialog::backdrop { background: oklch(from var(--n2) l c h / .4); }
""")
Dialog.variant('drawer', r"""
--_slide: 100%; --drawer-size: 24rem; --face: var(--n-3); inset: 0; inset-inline-start: auto; overflow-y: auto;
block-size: 100dvh; max-block-size: none; inline-size: var(--drawer-size); max-inline-size: none; margin: 0;
border-radius: 0; border-block: 0; border-inline-end: 0; translate: var(--_slide) 0;
&:dir(rtl) { --_slide: -100%; }
&[drawer=start] { --_slide: -100%; inset-inline: 0 auto; border-inline: 0; border-inline-end: 1px solid var(--n-1); }
&[drawer=start]:dir(rtl) { --_slide: 100%; }
&[open] { translate: none; @starting-style { translate: var(--_slide) 0; } }
@media (width < 40rem) { &, &[drawer=start] { inline-size: 100%; border: 0; } }
""", layer='components', doc='Slides in from the end edge and runs the full height, `--drawer-size` wide, or the whole screen below 40rem, on the page colour, scrolling when its content is taller; `start` slides in from the other edge.')

Dialog.drawer [components] :is(dialog)[drawer]: Slides in from the end edge and runs the full height, `--drawer-size` wide, or the whole screen below 40rem; `start` slides in from the other edge.

A dialog that htmx fills opens itself. A button that targets the dialog's body with `hx-get` needs no `command`, and the dialog shows nothing stale: it opens once the new content has arrived. A swap of the dialog itself, such as a closed one sent out of band to close it, leaves it closed.

In [ ]:
Dialog.rule('fill', on={('&', ':after-swap'): "if (e.target !== el && !el.open) el.showModal()"}, layer='components',
    doc='A closed dialog opens when htmx swaps content into it; a swap of the dialog itself leaves it as it is.')

Dialog.fill [components] dialog: A closed dialog opens when htmx swaps content into it; a swap of the dialog itself leaves it as it is.

A confirmation with its trigger. `aria-labelledby` connects the dialog to its heading; simply putting a heading inside a dialog does not name it. The content is an ordinary stack with a cluster of buttons.

In [ ]:
@example(Dialog)
def _dialog_example():
    id, title = unqid(), unqid()
    return Cluster(Button('Delete project', type='button', danger=True, command='show-modal', commandfor=id),
        Dialog(Stack(H2('Delete this project?', id=title), P('Everything in it is removed. This cannot be undone.'),
            Cluster(Button('Cancel', type='button', secondary=True, command='close', commandfor=id, autofocus=True),
                Button('Delete', type='button', danger=True, command='close', commandfor=id), gap='var(--space-2)')),
            id=id, aria_labelledby=title))

In [ ]:
Dialog.demo(height='16rem')

```html
<div cluster>
<button type="button" danger command="show-modal" commandfor="_fyi3GY1BRbyLLfZZ9Gfl9Q">Delete project</button><dialog aria-labelledby="_KJ5nID89T3aw4K4laft6Ag" id="_fyi3GY1BRbyLLfZZ9Gfl9Q">    <div stack>
      <h2 id="_KJ5nID89T3aw4K4laft6Ag">Delete this project?</h2>
      <p>Everything in it is removed. This cannot be undone.</p>
      <div cluster style="--gap: var(--space-2);">
<button type="button" secondary command="close" commandfor="_fyi3GY1BRbyLLfZZ9Gfl9Q" autofocus>Cancel</button><button type="button" danger command="close" commandfor="_fyi3GY1BRbyLLfZZ9Gfl9Q">Delete</button>      </div>
    </div>
</dialog></div>

```

## Drawer

A drawer is a dialog that slides in from an edge and runs the full height: `drawer=True` for the right edge, `drawer='start'` for the left. It is `--drawer-size` wide, 24rem unless the app sets the token, and below 40rem it fills the screen, since a sliver of page beside it is no use on a phone. It scrolls when its content is taller than the screen. It sits on the page colour rather than the dialog's raised surface, because a drawer is a region of the page more than a floating panel, and cards inside it need to stand out from it. Everything else, opening, closing, the backdrop, and the inert page, is the dialog's. A drawer also closes on a click outside it, with `closedby=\"any\"`, in browsers that support that attribute; elsewhere its button still closes it:

In [ ]:
demo(Cluster(Button('Filters', type='button', secondary=True, command='show-modal', commandfor='filters'),
    Button('Navigation', type='button', secondary=True, command='show-modal', commandfor='nav-drawer'),
    Dialog(Stack(H2('Filters', id='filters-title'), P('Narrow the list.'),
        Button('Done', type='button', command='close', commandfor='filters')),
        id='filters', aria_labelledby='filters-title', drawer=True, closedby='any'),
    Dialog(Stack(H2('Navigation', id='nav-title'), P('Sections of the app.'),
        Button('Close', type='button', secondary=True, command='close', commandfor='nav-drawer')),
        id='nav-drawer', aria_labelledby='nav-title', drawer='start', closedby='any')), height='12rem')

```html
<div cluster>
<button type="button" secondary command="show-modal" commandfor="filters">Filters</button><button type="button" secondary command="show-modal" commandfor="nav-drawer">Navigation</button><dialog aria-labelledby="filters-title" drawer closedby="any" id="filters">    <div stack>
      <h2 id="filters-title">Filters</h2>
      <p>Narrow the list.</p>
<button type="button" command="close" commandfor="filters">Done</button>    </div>
</dialog><dialog aria-labelledby="nav-title" drawer="start" closedby="any" id="nav-drawer">    <div stack>
      <h2 id="nav-title">Navigation</h2>
      <p>Sections of the app.</p>
<button type="button" secondary command="close" commandfor="nav-drawer">Close</button>    </div>
</dialog></div>

```

## Tooltip

`Tooltip` pairs a control with a short, supplementary description. It adds `aria-describedby` without replacing an existing description, and returns the pair in one inline wrapper. The control still needs its own name; a tooltip is not a substitute for a label.

The description is real markup, shown by CSS alone. It appears while the pointer is over the control or the description, or while the control has keyboard focus, and goes 150ms after both are gone, which is the gap that bridges moving from the control onto the description. A click that focuses the control does not show it. It appears at once, and moving to another control swaps the two at once, with no fade between them, as the platform's own tooltips do. The control is the anchor: anchor positioning puts the description above it, centred, and flips it below at the top of the viewport. An engine without anchor positioning puts it above the control's start edge instead. It is not a popover, so it is not in the top layer: an ancestor with a `transform` or a `filter` clips it. Put interactive content in a `Popover`, not a tooltip.

In [ ]:
_tooltip_css = r"""
[tooltip] { display: inline-block; position: relative; }
[tooltip] > [role=tooltip] {
  --face: var(--n2); background: var(--face); color: var(--n-3); border: 0;
  padding: var(--space-3) var(--space-2); border-radius: var(--radius-1);
  inline-size: max-content; max-inline-size: min(20rem, calc(100vw - 2 * var(--space))); overflow-wrap: anywhere;
  font-size: var(--text-1); font-weight: var(--weight); line-height: var(--leading);
  position: fixed; inset: auto; position-area: block-start; position-try-fallbacks: flip-block; margin-block-end: 4px; z-index: 1;
  visibility: hidden; opacity: 0;
  transition: opacity var(--motion) var(--ease) 150ms, visibility 0s linear calc(var(--motion) + 150ms);
}
[tooltip]:is(:hover, :has(:focus-visible)) > [role=tooltip] { visibility: visible; opacity: 1; transition: none; }
:root:has([tooltip]:is(:hover, :has(:focus-visible))) [tooltip]:not(:hover, :has(:focus-visible)) > [role=tooltip] { transition: none; }
@supports not (position-area: center) {
  [tooltip] > [role=tooltip] { position: absolute; inset: auto auto 100% 0; }
}
"""

@hypelt(sel='[tooltip]', layer='components', css=_tooltip_css)
def Tooltip(
    trigger, # A control with its own accessible name
    text,    # Supplementary, non-interactive content
    id=None, # Description id; generated when omitted
    **attrs, # Attributes for the description
):
    "A control and its supplementary description, shown on hover or keyboard focus"
    id = id or unqid()
    describedby = ' '.join(filter(None, [trigger.get('aria-describedby'), id]))
    trigger(aria_describedby=describedby, style=' '.join(filter(None, [trigger.get('style'), f'anchor-name: --{id};'])))
    tip = Div(text, role='tooltip', id=id, **attrs)(style=' '.join(filter(None, [attrs.get('style'), f'position-anchor: --{id};'])))
    return Span(trigger, tip, tooltip=True)


In [ ]:
@example(Tooltip)
def _tooltip_example():
    return Cluster(Tooltip(Button('Archive', type='button', secondary=True), 'Move to the archive'),
        Tooltip(Button('Share', type='button', secondary=True), 'Copy a link'),
        Tooltip(Button('Delete', type='button', danger=True), 'This cannot be undone'))

In [ ]:
Tooltip.demo(height='7rem')

```html
<div cluster>
<span tooltip><button type="button" secondary aria-describedby="_ccVyN5AKTAe1vTJBWpzZtg" style="anchor-name: --_ccVyN5AKTAe1vTJBWpzZtg;">Archive</button>    <div role="tooltip" id="_ccVyN5AKTAe1vTJBWpzZtg" style="position-anchor: --_ccVyN5AKTAe1vTJBWpzZtg;">Move to the archive</div>
</span><span tooltip><button type="button" secondary aria-describedby="_wj4gTeBuQ_qa8-EdEUDPzg" style="anchor-name: --_wj4gTeBuQ_qa8-EdEUDPzg;">Share</button>    <div role="tooltip" id="_wj4gTeBuQ_qa8-EdEUDPzg" style="position-anchor: --_wj4gTeBuQ_qa8-EdEUDPzg;">Copy a link</div>
</span><span tooltip><button type="button" danger aria-describedby="_7PRi36XgRsOM4k38wOrApg" style="anchor-name: --_7PRi36XgRsOM4k38wOrApg;">Delete</button>    <div role="tooltip" id="_7PRi36XgRsOM4k38wOrApg" style="position-anchor: --_7PRi36XgRsOM4k38wOrApg;">This cannot be undone</div>
</span></div>

```

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()